# 10 — Modelos finales y predicciones sobre el test

Notebook de **cierre y entrega**. Reúne los modelos finales de los dos
componentes, con sus **hiperparámetros definitivos** (los re-tuneados por CV
temporal honesta en `retuning_cv_honesta.json`), los re-entrena sobre **todo el
train** y produce las **predicciones finales sobre el test** (≥2021):

- **Componente B (supervisado)** — rinde predicho (`rinde_pred_kgha`) del modelo
  campeón (elegido por **CV**, no por test), con la comparación de todos los
  modelos para justificar la elección.
- **Componente A (no supervisado)** — score de anomalía y flag `es_anomalo` del
  detector final (VAE `recon_prob` seed-ensemble).

Al final se arma **`predicciones_test.csv`** con el formato de entrega, y se deja
una función `predecir_entrega(...)` lista para re-apuntar al **test set oficial**
cuando se publique (24 h antes de la entrega).

> **Adaptación al test oficial:** la consigna dice que 24 h antes se publica un
> archivo de test con su estructura. Cuando llegue, sólo hay que pasarlo por
> `predecir_entrega(panel_nuevo)` (misma tubería de features y modelos ya
> entrenados) y entregar el CSV resultante. Acá lo demostramos sobre nuestro
> propio test temporal (≥2021), para el que sí tenemos el rinde real y podemos
> reportar métricas.

In [1]:
import sys, os, warnings, json
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev, latente
from modelos import (LinearRegressor, XGBoostRegressor, NeuralNetRegressor,
                     RandomForestRegressorModel, HistGBMRegressor, StackingRegressorModel)

CULTIVOS = ['soja', 'maiz']
DS_KW = dict(use_agro=True, enc_smooth=10.0)          # config final del pipeline
BEST = json.load(open('retuning_cv_honesta.json', encoding='utf-8'))
def best_of(cultivo, name, fallback=None):
    return BEST.get(cultivo, {}).get(name, {}).get('best_params', fallback or {})
print('re-tuning cargado para:', list(BEST))

re-tuning cargado para: ['soja', 'maiz']


## 1. Reconstrucción de los modelos finales

Cada modelo se instancia con sus hiperparámetros definitivos. El **campeón se
elige por el `cv_rmse`** (validación en train), de modo que la elección **no
depende del test** — el test sólo confirma.

In [2]:
def construir_modelos(cultivo):
    return {
        'Lineal':        LinearRegressor(**best_of(cultivo, 'linear', {'penalty': 'ridge', 'alpha': 10.0})),
        'Random Forest': RandomForestRegressorModel(**best_of(cultivo, 'rf'), random_state=42, n_jobs=-1),
        'HistGBM':       HistGBMRegressor(**best_of(cultivo, 'hist_gbm'), random_state=42),
        'XGBoost':       XGBoostRegressor(**best_of(cultivo, 'xgb'), random_state=42),
        'Red neuronal':  NeuralNetRegressor(**best_of(cultivo, 'nn', {'hidden_dims': (64, 32), 'dropout': 0.3}),
                                            max_epochs=250, patience=30, random_state=42),
    }

def campeon(cultivo):
    cvs = {n: BEST[cultivo][k].get('cv_rmse', np.inf)
           for n, k in [('Lineal','linear'), ('Random Forest','rf'), ('HistGBM','hist_gbm'),
                        ('XGBoost','xgb'), ('Red neuronal','nn')] if k in BEST.get(cultivo, {})}
    return min(cvs, key=cvs.get) if cvs else 'Random Forest'

for c in CULTIVOS:
    print(f'{c}: campeón por CV = {campeon(c)}')

soja: campeón por CV = Random Forest
maiz: campeón por CV = HistGBM


## 2. Comparación de todos los modelos en test (justifica el campeón)

Entrenamos cada modelo en el train y evaluamos en test. Reportamos RMSE/R²/sMAPE,
**skill vs. climatología** y marcamos el campeón elegido por CV.

In [3]:
def evaluar_cultivo(cultivo):
    ds = datos.prepare(cultivo, **DS_KW)
    yrs = ds.meta_train['campania_inicio'].values
    modelos = construir_modelos(cultivo)
    filas, preds = [], {}
    preds['media x depto'] = ev.pred_media_depto(ds)
    filas.append(ev.evaluar('media x depto', preds['media x depto'], ds))
    for nombre, m in modelos.items():
        p = m.fit(ds.X_train, ds.y_train).predict(ds.X_test)
        preds[nombre] = p
        filas.append(ev.evaluar(nombre, p, ds))
    # stacking (usa años para OOF temporal)
    stk = StackingRegressorModel(**best_of(cultivo, 'stacking', {})).fit(
        ds.X_train, ds.y_train, years=yrs)
    preds['Stacking'] = stk.predict(ds.X_test)
    filas.append(ev.evaluar('Stacking', preds['Stacking'], ds))
    tabla = ev.tabla_comparativa(filas, ordenar_por='rmse')
    ref = ev.pred_media_depto(ds)
    tabla['skill'] = [ev.skill_score(ds.y_test, preds[m], ref) if m in preds else np.nan
                      for m in tabla['modelo']]
    return ds, preds, tabla

resultados = {}
for c in CULTIVOS:
    ds, preds, tabla = evaluar_cultivo(c)
    resultados[c] = (ds, preds, tabla)
    print(f'\n===== {c} (campeón CV: {campeon(c)}) =====')
    print(tabla.to_string(index=False))

[data] dedup panel: 27865 -> 20672 filas (7193 duplicados espurios por lat/lon eliminados)



===== soja (campeón CV: Random Forest) =====
       modelo     mae    rmse    r2   mape  smape  skill
Random Forest 465.173 592.845 0.401 25.656 21.874  0.163
      XGBoost 470.120 606.071 0.374 26.501 22.039  0.144
      HistGBM 480.686 616.425 0.353 27.162 22.494  0.130
     Stacking 507.702 649.336 0.282 29.705 23.498  0.083
       Lineal 507.692 660.554 0.257 29.870 23.667  0.067
 Red neuronal 520.040 664.559 0.248 29.962 24.015  0.062
media x depto 593.307 708.357 0.145 30.295 27.169  0.000
[data] dedup panel: 27865 -> 20672 filas (7193 duplicados espurios por lat/lon eliminados)



===== maiz (campeón CV: HistGBM) =====
       modelo       mae      rmse     r2   mape  smape  skill
Random Forest 1,143.872 1,535.628  0.414 30.335 24.416  0.140
      HistGBM 1,292.238 1,709.344  0.274 34.021 26.714  0.043
      XGBoost 1,304.034 1,720.795  0.265 34.268 27.039  0.036
media x depto 1,500.686 1,785.230  0.208 30.634 31.662  0.000
     Stacking 1,499.564 1,955.928  0.050 39.753 29.364 -0.096
       Lineal 1,519.667 1,969.778  0.036 39.019 29.330 -0.103
 Red neuronal 1,596.273 2,019.356 -0.013 40.585 31.026 -0.131


Diebold–Mariano del campeón vs. el resto (¿la ventaja es significativa?).

In [4]:
for c in CULTIVOS:
    ds, preds, tabla = resultados[c]
    camp = campeon(c)
    print(f'--- {c}: {camp} vs. resto ---')
    for m in preds:
        if m == camp: continue
        dm = ev.diebold_mariano(ds.y_test, preds[camp], preds[m], loss='se')
        sig = 'sig.' if dm['p_value'] < 0.05 else 'n.s.'
        print(f'  vs {m:16s} DM={dm["dm"]:+.2f}  p={dm["p_value"]:.3f}  [{sig}]')

--- soja: Random Forest vs. resto ---
  vs media x depto    DM=-9.64  p=0.000  [sig.]
  vs Lineal           DM=-7.97  p=0.000  [sig.]
  vs HistGBM          DM=-2.95  p=0.003  [sig.]
  vs XGBoost          DM=-2.64  p=0.008  [sig.]
  vs Red neuronal     DM=-7.95  p=0.000  [sig.]
  vs Stacking         DM=-7.99  p=0.000  [sig.]
--- maiz: HistGBM vs. resto ---
  vs media x depto    DM=-1.83  p=0.068  [n.s.]
  vs Lineal           DM=-11.75  p=0.000  [sig.]
  vs Random Forest    DM=+11.91  p=0.000  [sig.]
  vs XGBoost          DM=-1.22  p=0.222  [n.s.]
  vs Red neuronal     DM=-15.51  p=0.000  [sig.]
  vs Stacking         DM=-16.83  p=0.000  [sig.]


## 3. Componente A: score de anomalía final sobre el test

El detector final es el **VAE `recon_prob` en seed-ensemble** (Componente A). Su
`anomaly_score` (continuo) va alineado fila a fila con el test del predictor y es
la señal principal.

**Caveat del flag binario.** El umbral calibrado en train marca casi TODO el test
como anómalo: entre 2021 y 2024 el score sube por *distribution shift* (años fuera
del rango de train), no porque cada campaña sea extrema. Para la entrega, definimos
`es_anomalo` como el **top-10% más anómalo por score dentro del set evaluado**
(ranking relativo, sin usar etiquetas) — así el flag señala las campañas *más*
anómalas de forma útil, y el score continuo queda para el ranking fino.

In [5]:
CONTAM = 0.10                                          # fracción marcada como anómala
scores_A = {}
for c in CULTIVOS:
    vf = latente.vae_features(c)                       # cacheado en ../.latente_cache
    score = vf['score_test']
    thr = np.quantile(score, 1.0 - CONTAM)             # umbral relativo al set evaluado
    flag_te = (score >= thr).astype(int)
    scores_A[c] = (score, flag_te)
    print(f'{c}: {int(flag_te.sum())} campañas marcadas anómalas de {len(flag_te)} '
          f'(top-{int(CONTAM*100)}% por score)')

soja: 95 campañas marcadas anómalas de 942 (top-10% por score)
maiz: 116 campañas marcadas anómalas de 1160 (top-10% por score)


## 4. Archivo de entrega: `predicciones_test.csv`

Formato propuesto (adaptable a la estructura que publique la cátedra): una fila
por **departamento × campaña × cultivo** del test, con identificadores + la
predicción de rinde del campeón + el score/flag de anomalía. Se incluye
`rinde_real_kgha` **solo como referencia** para nuestra evaluación (en el test
oficial ciego no estaría).

In [6]:
bloques = []
for c in CULTIVOS:
    ds, preds, tabla = resultados[c]
    camp = campeon(c)
    score, flag = scores_A[c]
    out = ds.meta_test[['provincia', 'departamento', 'campania']].copy()
    out.insert(0, 'cultivo', c)
    out['rinde_real_kgha'] = ds.y_test                 # referencia (no va en test ciego)
    out['rinde_pred_kgha'] = np.round(preds[camp], 1)
    out['modelo'] = camp
    out['anomaly_score'] = np.round(score, 4)
    out['es_anomalo'] = flag.astype(int)
    bloques.append(out)

entrega = pd.concat(bloques, ignore_index=True)
entrega.to_csv('predicciones_test.csv', index=False, encoding='utf-8')
print('guardado: predicciones_test.csv', entrega.shape)
entrega.head(8)

guardado: predicciones_test.csv (2102, 9)


,cultivo,provincia,departamento,campania,rinde_real_kgha,rinde_pred_kgha,modelo,anomaly_score,es_anomalo
0,soja,BUENOS AIRES,25 De Mayo,2021/22,"2,817.000","2,955.900",Random Forest,141.844,0
1,soja,BUENOS AIRES,25 De Mayo,2022/23,"1,549.000","2,460.000",Random Forest,"2,904.066",1
2,soja,BUENOS AIRES,25 De Mayo,2023/24,"2,501.000","3,104.100",Random Forest,135.536,0
3,soja,BUENOS AIRES,25 De Mayo,2024/25,"2,800.000","2,856.700",Random Forest,125.520,0
4,soja,BUENOS AIRES,9 De Julio,2021/22,"3,028.000","3,282.300",Random Forest,115.754,0
5,soja,BUENOS AIRES,9 De Julio,2022/23,"1,370.000","2,589.900",Random Forest,"3,851.969",1
6,soja,BUENOS AIRES,9 De Julio,2023/24,"3,265.000","3,191.300",Random Forest,193.459,0
7,soja,BUENOS AIRES,9 De Julio,2024/25,"2,963.000","3,078.100",Random Forest,133.363,0


In [7]:
# Métricas finales del archivo entregado (por cultivo), para el informe
for c in CULTIVOS:
    sub = entrega[entrega.cultivo == c]
    m = ev.metricas(sub.rinde_real_kgha.values, sub.rinde_pred_kgha.values)
    print(f'{c:5s} ({sub.modelo.iloc[0]:13s}): '
          f'RMSE={m["rmse"]:.0f}  R2={m["r2"]:.3f}  sMAPE={m["smape"]:.1f}%')

soja  (Random Forest): RMSE=593  R2=0.401  sMAPE=21.9%
maiz  (HistGBM      ): RMSE=1709  R2=0.274  sMAPE=26.7%


## 5. Predicción sobre el test set OFICIAL (cuando se publique)

Cuando la cátedra publique el archivo de test (24 h antes), guardarlo como panel
con el **mismo esquema** que `panel_union.parquet` (columnas de clima/NDVI/ERA5 +
identificadores) y correr la función de abajo. Entrena los modelos sobre **todo**
nuestro panel conocido y predice sobre las filas nuevas, devolviendo el CSV en el
mismo formato.

In [8]:
def predecir_entrega(panel_nuevo, cultivos=CULTIVOS, salida='predicciones_oficial.csv'):
    """Predice rinde + anomalía sobre un panel de test externo.

    panel_nuevo: DataFrame con el MISMO esquema que panel_union (features de clima/
    NDVI/ERA5 + provincia/departamento/campania/campania_inicio/cultivo). Entrena
    sobre todo el panel conocido (train = histórico) y predice las filas nuevas.
    """
    panel_base = datos.load_panel()
    te_start = int(panel_nuevo['campania_inicio'].min())
    panel = pd.concat([panel_base, panel_nuevo], ignore_index=True)
    bloques = []
    for c in cultivos:
        ds = datos.build_reg_dataset(panel, c, test_start=te_start,
                                     train_end=te_start - 1, **DS_KW)
        camp = campeon(c)
        modelo = construir_modelos(c)[camp]
        pred = modelo.fit(ds.X_train, ds.y_train).predict(ds.X_test)
        out = ds.meta_test[['provincia', 'departamento', 'campania']].copy()
        out.insert(0, 'cultivo', c)
        out['rinde_pred_kgha'] = np.round(pred, 1)
        out['modelo'] = camp
        bloques.append(out)
    entrega = pd.concat(bloques, ignore_index=True)
    entrega.to_csv(salida, index=False, encoding='utf-8')
    return entrega

# Ejemplo de uso (descomentar cuando exista el archivo oficial):
# panel_oficial = pd.read_parquet('test_oficial.parquet')
# predecir_entrega(panel_oficial)
print('predecir_entrega() lista para el test oficial.')

predecir_entrega() lista para el test oficial.


## Conclusión

- El **campeón se elige por `cv_rmse`** (validación en train), no por test:
  **Random Forest en soja** y **HistGBM en maíz**. Nota honesta: en maíz, RF le
  gana a HistGBM en el test (ver la tabla), pero HistGBM ganó la CV — respetamos
  la elección por CV para **no decidir sobre el test**. Es el protocolo correcto y
  la comparación + Diebold–Mariano quedan transparentes para el lector.
- El **detector final** (VAE seed-ensemble) aporta el `anomaly_score` continuo por
  campaña; el flag `es_anomalo` marca el top-10% por score (ver caveat de
  distribution shift), integrando ambos componentes en una sola salida.
- `predicciones_test.csv` es el entregable; `predecir_entrega()` re-genera el CSV
  sobre el test oficial en cuanto se publique, sin tocar el resto del pipeline.